# Skin Condition Classification
Educational five-class image classification; not a medical diagnosis.

## 1. Import Libraries

In [ ]:
from pathlib import Path
import sys, json, numpy as np, matplotlib.pyplot as plt
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import tensorflow as tf
from config import *
from src.dataset import load_datasets
from src.models import build_model, enable_fine_tuning
from src.metrics import plot_history, evaluate_and_save
from src.utils import set_seed
set_seed(SEED)

## 2. Check GPU

In [ ]:
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU') or 'None; using CPU')

## 3. Configuration

In [ ]:
print({'seed': SEED, 'image_size': IMG_SIZE, 'batch_size': BATCH_SIZE, 'classes': CLASS_NAMES})

## 4. Dataset Audit
Run this before preparation. Ambiguous COCO images and extra oily/dry labels are excluded, not guessed.

In [ ]:
from scripts.audit_dataset import run_audit
eligible_images, excluded_images, audit_summary = run_audit()

## 5. Dataset Loading
First run `python scripts/prepare_dataset.py` in a terminal, then load the fixed train/validation/test split.

In [ ]:
train_ds, val_ds, test_ds = load_datasets()

## 6. Show Sample Images

In [ ]:
images, labels = next(iter(train_ds))
plt.figure(figsize=(10, 8))
for i in range(min(9, len(images))):
    plt.subplot(3, 3, i + 1); plt.imshow(images[i].numpy().astype('uint8'))
    plt.title(CLASS_NAMES[int(labels[i])]); plt.axis('off')
plt.tight_layout()

## 7. Pre-processing
The loader resizes to 224×224. Each model applies its ImageNet normalization internally.

In [ ]:
print('Batch shape after resize:', images.shape)
print('Raw pixel range:', float(tf.reduce_min(images)), float(tf.reduce_max(images)))

## 8. Data Augmentation
Mild flip, rotation, zoom, translation, and contrast run only while training.

In [ ]:
from src.preprocessing import make_augmentation
augmented = make_augmentation(SEED)(images[:1], training=True)
plt.imshow(tf.cast(tf.clip_by_value(augmented[0], 0, 255), tf.uint8)); plt.axis('off')

## 9. Build Transfer Learning Model

In [ ]:
MODEL_NAME = 'mobilenetv2'  # also efficientnetb0 or resnet50
model, backbone = build_model(MODEL_NAME)
model.summary()
assert model.output_shape[-1] == 5

## 10. Train Model (Stage 1)
The backbone is frozen. Use at least 10 epochs for the actual assignment; this cell does not touch the test set.

In [ ]:
history_stage1 = model.fit(train_ds, validation_data=val_ds, epochs=INITIAL_EPOCHS)

## 11. Plot Accuracy

In [ ]:
plt.plot(history_stage1.history['accuracy'], label='Training Accuracy')
plt.plot(history_stage1.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()

## 12. Plot Loss

In [ ]:
plt.plot(history_stage1.history['loss'], label='Training Loss')
plt.plot(history_stage1.history['val_loss'], label='Validation Loss')
plt.title('Model Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()

## 13. Fine Tuning (Stage 2)

In [ ]:
enable_fine_tuning(model, backbone, unfreeze_last=30, learning_rate=FINE_TUNE_LEARNING_RATE)
history_stage2 = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS)

## 14. Final Evaluation
Only now is the held-out test set used.

In [ ]:
metrics = evaluate_and_save(model, test_ds, CLASS_NAMES, RESULTS_DIR / MODEL_NAME)
metrics

## 15. Confusion Matrix

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(RESULTS_DIR / MODEL_NAME / 'confusion_matrix.png')))

## 16. Precision / Recall / F1-score

In [ ]:
print((RESULTS_DIR / MODEL_NAME / 'classification_report.txt').read_text())

## 17. Predict New Image
For consistent class order, save/train through `train.py`, then use:
`python predict.py --image example.jpg --model artifacts/mobilenetv2/best_model.keras`

In [ ]:
# Example after setting IMAGE_PATH:
# image = tf.keras.utils.load_img(IMAGE_PATH, target_size=(IMG_SIZE, IMG_SIZE))
# probabilities = model.predict(np.expand_dims(tf.keras.utils.img_to_array(image), 0))[0]
# dict(zip(CLASS_NAMES, probabilities.tolist()))

## 18. Compare Models
Train all three with the scripts, then read the measured comparison—never fill in fake results.

In [ ]:
import pandas as pd
comparison_path = RESULTS_DIR / 'model_comparison.csv'
pd.read_csv(comparison_path) if comparison_path.exists() else 'Run: python compare_models.py after training models'